# Camera-trap live demonstration
Run both cells with the team Drive mounted. Video mode samples up to five frames at one-second intervals.

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CameraTrap_Phase2')
PROJECT = Path.cwd()
if not (PROJECT/'runtime/live.py').is_file():
    PROJECT = Path(tempfile.mkdtemp())/'project'
    subprocess.run(['git','clone','--quiet','--depth','1','--branch','project','https://github.com/salehdev3/Capstone_5.git',str(PROJECT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt')],check=True)
sys.path.insert(0,str(PROJECT/'runtime'))
from live import cases, execute, initialize, video_cases, team_test_cases, cct_test_cases
pipeline, decision = initialize(ROOT)
print('Loaded integrated release:', pipeline.policy_sha256)


In [ ]:
import pandas as pd
from IPython.display import display, Image as DisplayImage, Video

DEMO_INPUT = 'cct_test' #@param ['cct_test','team_test','photos','video']

if DEMO_INPUT == 'cct_test':
    batch = cct_test_cases(ROOT)
elif DEMO_INPUT == 'team_test':
    batch = team_test_cases(ROOT)
else:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('Select at least one file.')
    upload_dir = Path(tempfile.mkdtemp())
    if DEMO_INPUT == 'video':
        if len(uploaded) != 1:
            raise ValueError('Select one video.')
        name, raw = next(iter(uploaded.items()))
        video_path = upload_dir / Path(name).name
        video_path.write_bytes(raw)
        batch = video_cases(video_path, upload_dir/'frames')
        print('Video mode samples up to five frames, one per second.')
    else:
        batch = []
        for index, (name, raw) in enumerate(uploaded.items()):
            path = upload_dir / f'{index}.jpg'
            path.write_bytes(raw)
            batch.append((path, dict(id=f'upload-{index}', event=f'upload-{index}')))

print('Images selected:', len(batch))
dest, result = execute(pipeline, batch, ROOT/'presentation_demos')
display(pd.DataFrame(result['events']))
display(pd.DataFrame([
    {'id': frame['id'], 'animals': frame['count'],
     'inference_seconds': frame['timing']['total']}
    for frame in result['frames']
]))
for index, frame in enumerate(result['frames'], 1):
    display(DisplayImage(filename=str(dest/f'frame_{index:02}.jpg'), width=720))
display(Video(filename=str(dest/'Demo_Fallback.mp4'), embed=True))
print('HTTP response and recorded replay:', dest)
